# Spacetime Sentinels: reproducible walkthrough

This notebook follows the six required steps of the Qiskit Fall Fest 2026 Advanced-track challenge
(error detection with spacetime codes) and shows how every number used by the game is produced.

| Step | Section |
|---|---|
| 1. Build the payloads | §1 |
| 2. Fix the reference | §2 |
| 3. Add checks with qiskit-paulice | §3, §4 |
| 4. Run under noise | §5 |
| 5. Measure | §5, §6 |
| 6. Compare | §7 |

Runtime: about two minutes on a laptop CPU.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, qiskit, qiskit_aer, qiskit_paulice
from qiskit.quantum_info import Statevector, StabilizerState, Clifford
from sentinels import PayloadSpec, get_profile, checked_payload, evaluate_configuration, reference_state
from sentinels.checks import checked_prefixes
from sentinels.experiment import choose_sites
from sentinels.fidelity import exact_evaluation, noiseless_false_alarms
from sentinels.sampling import sample_postselected
from importlib.metadata import version
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__, "| qiskit-paulice", version("qiskit-paulice"))

qiskit 2.5.2 | qiskit-aer 0.17.2 | qiskit-paulice 0.2.1


## 1. Payloads
A 6-qubit brickwork of CZ gates with random `sx`/`s` single-qubit Cliffords (the family of the IBM tutorial).
The T-doped family inserts 0, 1, 2, 4, 8 or 16 T gates into the same skeleton.

In [2]:
spec = PayloadSpec(num_qubits=6, depth=12, seed=7)
payload = spec.build()
print("width", payload.num_qubits, "| depth", payload.depth(), "| two-qubit depth",
      payload.depth(lambda i: i.operation.num_qubits == 2), "| gates", dict(payload.count_ops()))
payload.draw("text", fold=-1)

width 6 | depth 40 | two-qubit depth 12 | gates {'sx': 81, 's': 32, 'cz': 30, 'h': 6}


┌───┐   ┌────┐┌───┐ ┌────┐┌────┐      ┌────┐┌───┐ ┌────┐┌───┐ ┌────┐                  ┌────┐┌───┐                  ┌────┐┌────┐┌────┐            ┌───┐ ┌────┐┌───┐                        ┌────┐┌────┐┌────┐                             
q_0: ┤ H ├─■─┤ √X ├┤ S ├─┤ √X ├┤ √X ├──■───┤ √X ├┤ S ├─┤ √X ├┤ S ├─┤ √X ├──────────────■───┤ √X ├┤ S ├──────────────■───┤ √X ├┤ √X ├┤ √X ├────────■───┤ S ├─┤ √X ├┤ S ├────────────────────■───┤ √X ├┤ √X ├┤ √X ├─────────────────────────────
     ├───┤ │ ├────┤├───┤ ├────┤└────┘  │   ├────┤└───┘ └────┘└───┘ └────┘┌────┐┌────┐  │   ├───┬┘├───┴┐     ┌────┐  │   ├────┤└────┘└────┘        │   ├───┴┐├───┬┘└───┘            ┌────┐  │   ├────┤├───┬┘└────┘                 ┌────┐┌────┐
q_1: ┤ H ├─■─┤ √X ├┤ S ├─┤ √X ├──■─────■───┤ √X ├────────────────────■───┤ √X ├┤ √X ├──■───┤ S ├─┤ √X ├──■──┤ √X ├──■───┤ √X ├──────────────■─────■───┤ √X ├┤ S ├──────────────■───┤ √X ├──■───┤ √X ├┤ S ├────────────────────■───┤ √X ├┤ √X ├
     ├───┤   ├────┤└───┘ └────┘  │   ┌────┐├────┤      ┌────┐┌────┐  │   └────┘├────┤┌────┐└───┘ └────┘  │  ├────┤┌───┐ ├────┤      ┌────┐  │   ┌────┐└────┘└───┘ ┌────┐┌───┐  │   ├───┬┘┌────┐└────┘└───┘       ┌────┐┌───┐  │   ├────┤└────┘
q_2: ┤ H ├─■─┤ √X ├──────────────■───┤ √X ├┤ √X ├──■───┤ √X ├┤ √X ├──■─────■───┤ √X ├┤ √X ├──────────────■──┤ √X ├┤ S ├─┤ √X ├──■───┤ √X ├──■───┤ √X ├────────■───┤ √X ├┤ S ├──■───┤ S ├─┤ √X ├──────────────■───┤ √X ├┤ S ├──■───┤ √X ├──────
     ├───┤ │ ├────┤      ┌────┐      └────┘└────┘  │   ├────┤└────┘┌───┐   │   ├───┬┘├────┤      ┌────┐┌───┐└────┘└───┘ └────┘  │   ├────┤┌────┐└────┘┌────┐  │   ├────┤├───┤┌────┐└───┘ ├────┤┌───┐ ┌────┐  │   ├────┤└───┘┌────┐├────┤      
q_3: ┤ H ├─■─┤ √X ├──■───┤ √X ├────────────────────■───┤ √X ├──■───┤ S ├───■───┤ S ├─┤ √X ├──■───┤ √X ├┤ S ├────────────────────■───┤ √X ├┤ √X ├──■───┤ √X ├──■───┤ √X ├┤ S ├┤ √X ├──■───┤ √X ├┤ S ├─┤ √X ├──■───┤ √X ├──■──┤ √X ├┤ √X ├──────
     ├───┤   ├────┤  │   ├────┤┌───┐ ┌────┐            └────┘  │   ├───┴┐      ├───┤ ├────┤  │   ├────┤├───┤      ┌───┐             └────┘└────┘  │   ├────┤      ├────┤├───┤└────┘  │   ├────┤└───┘ ├────┤      └────┘  │  ├────┤└────┘      
q_4: ┤ H ├─■─┤ √X ├──■───┤ √X ├┤ S ├─┤ √X ├──■─────────────────■───┤ √X ├──■───┤ S ├─┤ √X ├──■───┤ √X ├┤ S ├──■───┤ S ├───────────────────────────■───┤ √X ├──■───┤ √X ├┤ S ├────────■───┤ √X ├──■───┤ √X ├──────────────■──┤ √X ├────────────
     ├───┤ │ ├────┤┌────┐├────┤├───┤ ├────┤  │   ┌────┐┌───┐ ┌────┐├───┬┘  │   ├───┴┐├───┬┘┌────┐├───┬┘└───┘  │   ├───┴┐┌───┐ ┌────┐┌────┐            └────┘  │   ├───┬┘└───┘            └────┘  │   ├───┬┘┌────┐┌────┐     └────┘            
q_5: ┤ H ├─■─┤ √X ├┤ √X ├┤ √X ├┤ S ├─┤ √X ├──■───┤ √X ├┤ S ├─┤ √X ├┤ S ├───■───┤ √X ├┤ S ├─┤ √X ├┤ S ├────────■───┤ √X ├┤ S ├─┤ √X ├┤ √X ├────────────────────■───┤ S ├──────────────────────────■───┤ S ├─┤ √X ├┤ √X ├───────────────────────
     └───┘   └────┘└────┘└────┘└───┘ └────┘      └────┘└───┘ └────┘└───┘       └────┘└───┘ └────┘└───┘            └────┘└───┘ └────┘└────┘                        └───┘                              └───┘ └────┘└────┘

## 2. Reference and fidelity measure
* Clifford payload: the ideal output is a stabilizer state, computed by stabilizer simulation (`StabilizerState`).
* T-doped payload: statevector simulation (`Statevector`), cheap at 6 qubits.

**Fidelity measure:** the state fidelity $F=\langle\psi|\sigma|\psi\rangle$ between the ideal state and the
post-selected data state $\sigma$, because it is sensitive to every error, including phase errors that a
computational-basis distance would miss, and for Clifford states it equals the stabilizer average
$2^{-n}\sum_{G\in S}\mathrm{Tr}(\sigma G)$ that direct fidelity estimation measures.

In [3]:
stab = StabilizerState(Clifford(payload))
psi = Statevector(payload)
print("stabilizer generators:", stab.clifford.to_labels(mode="S"))
print("stabilizer and statevector agree:", np.allclose(Statevector(stab.clifford.to_circuit()).probabilities(), psi.probabilities()))

stabilizer generators: ['-IIZXYZ', '+ZYZXII', '-ZXZYYX', '+YYIYYX', '-YZYZXY', '-IYYYII']
stabilizer and statevector agree: True


## 3. Checks with qiskit-paulice
`add_pauli_checks` returns one circuit per number of checks. Each check is an ancilla that applies
controlled-Pauli gates to one target qubit at several times, then is measured; the syndrome is the
parity of the ancilla bit and some output bits (`check_support`).

In [4]:
prof = get_profile("qff-synthetic")
print(prof)
variants = checked_prefixes(spec, (0, 1, 2, 3), prof)
for v in variants:
    print(f"k={len(v.targets)}  targets={v.targets}  gamma={v.gamma if v.gamma is None else round(v.gamma,4)}  supports={v.supports}")

NoiseProfile(name='qff-synthetic', p2=0.01, p_ro=0.01, p_t=0.0, source='Challenge text: synthetic NoiseModel(gate_noise=0.01); 1% readout')
k=0  targets=()  gamma=2.0364  supports=()
k=1  targets=(0,)  gamma=1.6147  supports=((0, 1, 2, 3, 6),)
k=2  targets=(0, 1)  gamma=1.5281  supports=((0, 1, 2, 3, 6), (0, 7))
k=3  targets=(0, 1, 2)  gamma=1.2612  supports=((0, 1, 2, 3, 6), (0, 7), (1, 4, 8))
k=4  targets=(0, 1, 2, 3)  gamma=1.1778  supports=((0, 1, 2, 3, 6), (0, 7), (1, 4, 8), (0, 1, 2, 3, 5, 9))


In [5]:
cp = checked_payload(spec, (0, 2, 4), prof)
cp.circuit.draw("text", fold=-1, idle_wires=False)

global phase: 7π/4
                      ┌───┐   ┌────┐┌───┐┌────┐┌────┐                  ┌────┐┌───┐ ┌────┐┌───┐ ┌────┐               ┌────┐┌───┐                                                     ┌────┐┌────┐┌────┐                                                     ┌───┐┌────┐┌───┐                                      ┌───┐          ┌───┐┌────┐┌────┐┌────┐   ┌─┐   
       q_0: ───────■──┤ H ├─■─┤ √X ├┤ S ├┤ √X ├┤ √X ├──────────────■───┤ √X ├┤ S ├─┤ √X ├┤ S ├─┤ √X ├─────────────■─┤ √X ├┤ S ├──────────────────────────────────────────────────■──┤ √X ├┤ √X ├┤ √X ├────────────────────────────────────────■──────────■─┤ S ├┤ √X ├┤ S ├──────────────────────────────────■───┤ H ├────────■─┤ H ├┤ √X ├┤ √X ├┤ √X ├───┤M├───
            ┌───┐  │  └───┘ │ ├────┤├───┤├────┤└────┘              │   ├────┤└───┘ └────┘└───┘ └────┘┌────┐┌────┐ │ ├───┬┘├───┴┐                                         ┌────┐  │  ├────┤└────┘└────┘                                        │   ┌────┐ │ ├───┤└────┘└───┘                          ┌────┐  │   ├───┴┐┌───┐  │ └───┘└────┘├────┤├────┤   └╥┘┌─┐
       q_1: ┤ H ├──┼────────■─┤ √X ├┤ S ├┤ √X ├──────────────■─────■───┤ √X ├────────────────────■───┤ √X ├┤ √X ├─■─┤ S ├─┤ √X ├─────────────────────────────────────■───┤ √X ├──■──┤ √X ├──────────────────────────────────────────────■─────■───┤ √X ├─┼─┤ S ├──────────────────────────────────■──┤ √X ├──■───┤ √X ├┤ S ├──┼────────■───┤ √X ├┤ √X ├────╫─┤M├
            ├───┤  │          └────┘├───┤├───┬┘      ┌────┐  │   ┌────┐├────┤      ┌────┐┌────┐  │   └────┘└────┘   └───┘ └────┘                       ┌────┐┌────┐  │   ├────┤     └────┘┌───┐ ┌───┐ ┌───┐┌────┐┌───┐ ┌────┐   ┌────┐  │   ┌────┐└────┘ │ └───┘┌───┐         ┌───┐ ┌────┐┌───┐   │  ├───┬┘┌────┐└────┘├───┴┐ │ ┌───┐  │   ├────┤└┬─┬─┘    ║ └╥┘
       q_2: ┤ H ├──┼────────────■───┤ H ├┤ H ├───■───┤ √X ├──■───┤ √X ├┤ √X ├──■───┤ √X ├┤ √X ├──■───────────────────────────────────────────────────■─┤ √X ├┤ √X ├──■───┤ √X ├───────■───┤ S ├─┤ H ├─┤ S ├┤ √X ├┤ S ├─┤ √X ├─■─┤ √X ├──■───┤ √X ├───────┼───■──┤ H ├───────■─┤ H ├─┤ √X ├┤ S ├───■──┤ S ├─┤ √X ├──■───┤ √X ├─┼─┤ S ├──■───┤ √X ├─┤M├──────╫──╫─
            ├───┤  │            │   └───┘└───┘   │   ├────┤      ├────┤└────┘  │   ├────┤└────┘                                               ┌───┐  │ ├───┬┘├────┤      ├────┤┌───┐  │   └───┘ └───┘ └───┘└────┘└───┘ └────┘ │ ├────┤┌────┐└────┘┌────┐ │   │  ├───┴┐┌───┐ │ ├───┴┐└────┘├───┴┐┌───┐├───┴┐└────┘  │   ├────┤ │ └───┘┌────┐├────┤ └╥┘  ┌─┐ ║  ║ 
       q_3: ┤ H ├──┼────────────┼────────────────■───┤ √X ├──■───┤ √X ├────────■───┤ √X ├──────────────────────────────────────────────────■──┤ S ├──■─┤ S ├─┤ √X ├──■───┤ √X ├┤ S ├──┼───────────────────────────────────────■─┤ √X ├┤ √X ├──■───┤ √X ├─┼───■──┤ √X ├┤ S ├─┼─┤ √X ├──■───┤ √X ├┤ S ├┤ √X ├────────■───┤ √X ├─┼───■──┤ √X ├┤ √X ├──╫───┤M├─╫──╫─
            ├───┤  │            │   ┌───┐┌───┐       ├────┤  │   ├───┬┘            └────┘┌───┐ ┌────┐┌───┐ ┌────┐   ┌───┐          ┌───┐   │  ├───┴┐   ├───┤ ├────┤  │   ├────┤├───┤  │         ┌───┐                           └────┘└────┘  │   ├────┤ │      ├────┤├───┤ │ └────┘  │   ├────┤└───┘├────┤            └────┘ │   │  ├────┤└┬─┬─┘  ║   └╥┘ ║  ║ 
       q_4: ┤ H ├──┼────────■───┼───┤ H ├┤ H ├───■───┤ √X ├──■───┤ H ├───────────────■───┤ H ├─┤ √X ├┤ S ├─┤ √X ├─■─┤ H ├────────■─┤ H ├───■──┤ √X ├─■─┤ S ├─┤ √X ├──■───┤ √X ├┤ S ├──┼─────■───┤ S ├─────────────────────────────────────────■───┤ √X ├─┼───■──┤ √X ├┤ S ├─┼─────────■───┤ √X ├──■──┤ √X ├───────────────────┼───■──┤ √X ├─┤M├────╫────╫──╫──╫─
            ├───┤  │        │   │   └───┘└───┘   │   ├────┤┌────┐├───┴┐┌───┐ ┌────┐  │   └───┘ └────┘└───┘ └────┘ │ ├───┴┐┌───┐  │ ├───┴┐┌───┐└────┘ │ ├───┴┐├───┬┘┌────┐├───┬┘└───┘  │     │   ├───┴┐┌───┐┌────┐┌────┐                           └────┘ │   │  ├───┬┘└───┘ │             └────┘  │  ├───┬┘┌────┐┌────┐ ┌─┐   │      └────┘ └╥┘    ║    ║  ║  ║ 
       q_5: ┤ H ├──┼────────┼───┼────────────────■───┤ √X ├┤ √X ├┤ √X ├┤ S ├─┤ √X ├──┼────────────

## 4. T doping across checks
A hand-made or automatic check stays valid across a T gate only if it acts as I or Z on that wire, because T is diagonal.
`CheckedCircuit.uncovered_paulis` lists the wires where a Z error is invisible to every check: those are the
**Z-safe sites**. We verify validity directly: a valid check never fires on the noiseless circuit.

In [6]:
safe = cp.zsafe_sites()
print(f"{len(safe)} of {2*len(cp._cz_pos)} T sites are Z-safe for checks on {cp.targets}")
for policy in ("safe", "blind"):
    sites, ok, _ = choose_sites(cp, 4, policy)
    fa = noiseless_false_alarms(cp.doped(sites), cp.supports, cp.n)
    print(f"{policy:6s} sites={sites}  false-alarm rate per check on the IDEAL circuit = {np.round(fa, 4)}")

18 of 60 T sites are Z-safe for checks on (0, 2, 4)
safe   sites=[(23, 1), (1, 0), (2, 0), (9, 1)]  false-alarm rate per check on the IDEAL circuit = [0. 0. 0.]
blind  sites=[(23, 1), (26, 0), (3, 1), (9, 0)]  false-alarm rate per check on the IDEAL circuit = [0.1464 0.3232 0.1464]


## 5. Run under noise and measure
The noise model is the one announced in the challenge text (`NoiseModel(gate_noise=0.01)`, plus 1% readout),
applied identically by qiskit-paulice (to score checks) and by Qiskit Aer (to simulate). Every configuration
records the fidelity without checks, the post-selected fidelity, the kept fraction and the added cost.

In [7]:
rows = []
for targets in [(), (0, 2, 4), (0, 1, 2, 3)]:
    c = checked_payload(spec, targets, prof)
    for n_t, pol in [(0, "safe"), (4, "safe"), (4, "blind"), (4, "blind-retire")]:
        if not targets and pol != "safe":
            continue
        r = evaluate_configuration(c, n_t, pol, prof)
        rows.append(r)
        print(f"checks={str(targets):12s} T={n_t} {pol:12s} F0={r['F0']:.4f}  F_ps={r['F_ps']:.4f}  keep={r['keep']:.4f}  "
              f"R={r['R']:.4f}  ancillas={r['ancillas']} +2q={r['added_2q']} depth2q={r['depth_2q_bare']}->{r['depth_2q']} broken={r['broken']}")

checks=()           T=0 safe         F0=0.7225  F_ps=0.7225  keep=1.0000  R=0.7225  ancillas=0 +2q=0 depth2q=12->12 broken=[]
checks=()           T=4 safe         F0=0.7225  F_ps=0.7225  keep=1.0000  R=0.7225  ancillas=0 +2q=0 depth2q=12->12 broken=[]


checks=(0, 2, 4)    T=0 safe         F0=0.7225  F_ps=0.9131  keep=0.7046  R=0.7665  ancillas=3 +2q=9 depth2q=12->17 broken=[]


checks=(0, 2, 4)    T=4 safe         F0=0.7217  F_ps=0.9123  keep=0.7046  R=0.7658  ancillas=3 +2q=9 depth2q=12->17 broken=[]


checks=(0, 2, 4)    T=4 blind        F0=0.7225  F_ps=0.6290  keep=0.4700  R=0.4312  ancillas=3 +2q=9 depth2q=12->17 broken=[0, 1, 2]


checks=(0, 2, 4)    T=4 blind-retire F0=0.7225  F_ps=0.3853  keep=1.0000  R=0.3853  ancillas=3 +2q=9 depth2q=12->17 broken=[0, 1, 2]


checks=(0, 1, 2, 3) T=0 safe         F0=0.7225  F_ps=0.9224  keep=0.6518  R=0.7447  ancillas=4 +2q=15 depth2q=12->20 broken=[]


checks=(0, 1, 2, 3) T=4 safe         F0=0.7241  F_ps=0.9236  keep=0.6518  R=0.7457  ancillas=4 +2q=15 depth2q=12->20 broken=[]


checks=(0, 1, 2, 3) T=4 blind        F0=0.7225  F_ps=0.6044  keep=0.3674  R=0.3663  ancillas=4 +2q=15 depth2q=12->20 broken=[0, 2, 3]


checks=(0, 1, 2, 3) T=4 blind-retire F0=0.7225  F_ps=0.3967  keep=0.9089  R=0.3782  ancillas=4 +2q=15 depth2q=12->20 broken=[0, 2, 3]


## 6. Validation with the SamplerV2 primitive
The game uses exact density-matrix numbers. The same circuits sampled with the Aer `SamplerV2` primitive give the same
kept fraction and classical fidelity within shot noise.

In [8]:
sites, _, _ = choose_sites(cp, 4, "safe")
qc, bare = cp.doped(sites), cp.bare(sites)
psi_t, method = reference_state(bare)
ev = exact_evaluation(qc, cp.supports, cp.n, prof, psi_t)
sr = sample_postselected(qc, cp.supports, cp.n, prof, np.abs(psi_t)**2, shots=30_000)
print(f"reference: {method}")
print(f"kept fraction exact {ev.keep:.4f} | sampled {sr.keep:.4f} ± {sr.keep_stderr:.4f}")
print(f"classical fidelity exact {ev.hellinger:.4f} | sampled {sr.hellinger:.4f}")

reference: statevector
kept fraction exact 0.7046 | sampled 0.7053 ± 0.0026
classical fidelity exact 0.9981 | sampled 0.9973


## 7. Compare: the best trade-off
The full sweep over every legal move (`experiments/e0_game_grid.py`) is summarised in `results/data/best_moves_qff-synthetic.json`.

In [9]:
import json, pathlib
p = pathlib.Path("../results/data/best_moves_qff-synthetic.json")
if p.exists():
    for r in json.loads(p.read_text()):
        print(f"depth {r['depth']:2d}  T {r['n_t']:2d}  best checks {str(r['targets']):14s} {r['policy']:12s} "
              f"F0 {r['F0']:.3f} -> F_ps {r['F_ps']:.3f} keep {r['keep']:.3f}  gain x{r['gain']:.3f}  illegal moves {r['infeasible_moves']}")
else:
    print("Run `make grid figures` first.")

depth  6  T  0  best checks [0, 4]         none         F0 0.828 -> F_ps 0.914 keep 0.851  gain x1.018  illegal moves 0
depth  6  T  1  best checks [0, 4]         safe         F0 0.828 -> F_ps 0.914 keep 0.851  gain x1.018  illegal moves 0
depth  6  T  2  best checks [0, 4]         safe         F0 0.828 -> F_ps 0.914 keep 0.851  gain x1.018  illegal moves 0
depth  6  T  4  best checks [0, 4]         safe         F0 0.832 -> F_ps 0.914 keep 0.851  gain x1.013  illegal moves 0
depth  6  T  8  best checks [0, 4]         safe         F0 0.831 -> F_ps 0.911 keep 0.851  gain x1.011  illegal moves 0
depth  6  T 16  best checks [0, 4]         safe         F0 0.833 -> F_ps 0.911 keep 0.851  gain x1.009  illegal moves 2
depth 12  T  0  best checks [0, 3]         none         F0 0.722 -> F_ps 0.896 keep 0.750  gain x1.074  illegal moves 0
depth 12  T  1  best checks [0, 3]         safe         F0 0.722 -> F_ps 0.896 keep 0.750  gain x1.074  illegal moves 0
depth 12  T  2  best checks [0, 3]      

## 8. Play a round live
`sentinels play` runs the same computation live in the terminal. The web game at GitHub Pages plays back these
pre-computed Qiskit results for every legal move.

In [10]:
from sentinels.cli import main
main(["play", "--depth", "12", "--t", "4", "--targets", "0,2,4", "--policy", "safe"])


SPACETIME SENTINELS  |  6 data qubits, 12 CZ layers, 4 T gates  |  noise qff-synthetic (p2=0.01, p_ro=0.01)
Ancilla budget: 4. Choose data qubits 0..5 to guard.

Running qiskit-paulice + Qiskit Aer (exact density matrix)...


  ancillas 3   extra 2q gates 9   2q depth 12 -> 17
  fidelity without checks   F0   = 0.7217
  post-selected fidelity    F_ps = 0.9123
  kept-shot fraction        keep = 0.7046
  fidelity rate R = F_ps*sqrt(keep) = 0.7658   (bare 0.7217, gain x1.061)

  SCORE 766   (unprotected circuit scores 722)


0